# Telco Customer Churn - Exploratory Data Analysis (EDA)

### Business problem
This Data Science project seeks to predict which customers are likely to churn in the next billing cycle, enabling proactice retention interventions before revenue is lost.
### Stakeholder
Management, Sales and Marketing teams need this model to predict if a customer will churn or not to decide which customers to target with retention offers before they cancel.

## Cost of a Wrong Prediction
It will cost the company more if we predict a customer will not churn, but they churn than for the model to
predict that a customer will churn but they do not churn.
If average customer lifetime value is GHC1,200, a missed churner costs ~GHC1,200 in lost revenue. A false alarm costs only a retention offer of perhaps GHC50. The asymmetry justifies optimizing for recall over precision

## success metric
we should optimize for recall, because we need to rightly predict almost all customers that will churn

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Load data into jupyter environment
df = pd.read_csv("../data/raw/telco_customer_churn.csv")

`Exploratory Data Analytics`

In [3]:
print(
    f'Number of rows: {df.shape[0]}\n'
    f'Number of features: {df.shape[1]}'
)

Number of rows: 7043
Number of features: 21


In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
print(df.dtypes)

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


In [6]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


In [7]:
df.describe(exclude=[np.number]).T

,count,unique,top,freq
customerID,7043,7043,7590-VHVEG,1
gender,7043,2,Male,3555
Partner,7043,2,No,3641
Dependents,7043,2,No,4933
PhoneService,7043,2,Yes,6361
MultipleLines,7043,3,No,3390
InternetService,7043,3,Fiber optic,3096
OnlineSecurity,7043,3,No,3498
OnlineBackup,7043,3,No,3088
DeviceProtection,7043,3,No,3095


In [8]:
print('===NUMERICAL COLUMNS===')
numerical_columns = df.select_dtypes(include=[np.number]).columns.tolist()
for i, num_cols in enumerate(numerical_columns, 1):
    print(
        f"{i}. {num_cols:<20} | Min: {df[num_cols].min():<5} | Max: {df[num_cols].max()}"
    )

===NUMERICAL COLUMNS===
1. SeniorCitizen        | Min: 0     | Max: 1
2. tenure               | Min: 0     | Max: 72
3. MonthlyCharges       | Min: 18.25 | Max: 118.75


In [9]:
print("===CATEGORICAL COLUMNS===")
categorical_columns = df.select_dtypes(exclude=[np.number]).columns.tolist()
for i, cat_cols in enumerate(categorical_columns, 1):
    print(
        f"{i:<2}. {cat_cols:<30} | Uniques: {df[cat_cols].nunique():<4} | Examples: {df[cat_cols].unique()[:3]}"
    )

===CATEGORICAL COLUMNS===
1 . customerID                     | Uniques: 7043 | Examples: ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK']
2 . gender                         | Uniques: 2    | Examples: ['Female' 'Male']
3 . Partner                        | Uniques: 2    | Examples: ['Yes' 'No']
4 . Dependents                     | Uniques: 2    | Examples: ['No' 'Yes']
5 . PhoneService                   | Uniques: 2    | Examples: ['No' 'Yes']
6 . MultipleLines                  | Uniques: 3    | Examples: ['No phone service' 'No' 'Yes']
7 . InternetService                | Uniques: 3    | Examples: ['DSL' 'Fiber optic' 'No']
8 . OnlineSecurity                 | Uniques: 3    | Examples: ['No' 'Yes' 'No internet service']
9 . OnlineBackup                   | Uniques: 3    | Examples: ['Yes' 'No' 'No internet service']
10. DeviceProtection               | Uniques: 3    | Examples: ['No' 'Yes' 'No internet service']
11. TechSupport                    | Uniques: 3    | Examples: ['No' 'Yes' 'No int

In [10]:
print('===TARGET COLUMN DISTRIBUTION===')
print(df['Churn'].value_counts())
print(
    f'\nChurn rate: {round(df["Churn"].value_counts(normalize=True)["Yes"] * 100, 2)}%'
)

===TARGET COLUMN DISTRIBUTION===
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn rate: 26.54%


In [11]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [12]:
print(df.dtypes)

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object


`Data Quality Checks`

In [13]:
# checking for nulls
print("===NULL AUDIT===")
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
null_audit = pd.DataFrame(
    {
        'null_count': nulls,
        'null_pct': null_pct
    }
).query('null_count > 0')

print(null_audit if len(null_audit) > 0 else "No nulls found")

===NULL AUDIT===
              null_count  null_pct
TotalCharges          11      0.16


In [14]:
# Investigate null rows
null_rows = df[df['TotalCharges'].isnull()]
print(f'===NULL ROWS INVESTIGATIONS===')
print(null_rows[['tenure','MonthlyCharges', 'TotalCharges', 'Churn']])

===NULL ROWS INVESTIGATIONS===
      tenure  MonthlyCharges  TotalCharges Churn
488        0           52.55           NaN    No
753        0           20.25           NaN    No
936        0           80.85           NaN    No
1082       0           25.75           NaN    No
1340       0           56.05           NaN    No
3331       0           19.85           NaN    No
3826       0           25.35           NaN    No
4380       0           20.00           NaN    No
5218       0           19.70           NaN    No
6670       0           73.35           NaN    No
6754       0           61.90           NaN    No


In [15]:
df['SeniorCitizen'] = df['SeniorCitizen'].map({1: "Yes", 0: "No"}).astype('category')

In [16]:
df = df.dropna(how='any')

In [17]:
df = df.drop_duplicates(subset='customerID')

In [18]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [19]:
# cleaning checkpoint to confirm final state of our dataset
print(f'===POST CLEANING CHECKPOINT===')
print(f"Shape before cleaning (7043, 21)")
print(f'Shape after cleaning: {df.shape}')
print(f'\nRemaining nulls: {df.isnull().sum().sum()}')
print(f"Totalcharges dtype: {df['TotalCharges'].dtype}")
print(f"SeniorCitizen dtype: {df['SeniorCitizen'].dtype}")
print(f"\nChurn distribution after cleaning:")
print(f"{round(df['Churn'].value_counts(normalize=True)['Yes'] * 100, 2)}%")

===POST CLEANING CHECKPOINT===
Shape before cleaning (7043, 21)
Shape after cleaning: (7032, 21)

Remaining nulls: 0
Totalcharges dtype: float64
SeniorCitizen dtype: category

Churn distribution after cleaning:
26.58%


### EDA Insights
- There are 7,043 observations and 21 features - given 26.54% class imbalance, stratified k-fold cross validation will be used to ensure each fold maintins the churn ratio and produces reliable performance estimates.
- The dataset is heavily skewed with churn rate of only 26.54% - consider class imbalance in modelling
- TotalCharges column, upon inspection has been converted from a string dtype to float dtype using `pd.to_numeric` and `errors=coerce`. This introduced 11 null rows 
- After conversion, there are now 4 numeric columns and 17 categorical column
- The numeric columns are all within valid ranges `values >= 0`. 
- There are 17 categorical columns with very low cardinality, except for `customer_id` which is the unique identifier - one-hot encoding for features, and label encoding for target column.
- Senior Citizen has values 0 and 1, treat as category
- 11 rows dropped where TotalCharges was null. Investigation revealed all had tenure = 0, indicating new customers with no billing history. Imputation would introduce artifical pattterns, therefore rows have been dropped.
- Duplicate check on CustomerID = 0 duplicates found


In [20]:
output_path = Path('../data/interim/cleaned_telco_churn.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")
print(f"Final dataset shape: {df.shape}")

Cleaned dataset saved to: ..\data\interim\cleaned_telco_churn.csv
Final dataset shape: (7032, 21)
